In [3]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

C:\Users\Riyad\AppData\Roaming\Python\Python310\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\Riyad\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device: {device}")

 Device: cuda


In [5]:
# Load data
train_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/train_final.csv')
val_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/val_final.csv')
test_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/test_final.csv')

# Combine headline + content
train_df['text'] = train_df['headline'].fillna('') + ' ' + train_df['content'].fillna('')
val_df['text'] = val_df['headline'].fillna('') + ' ' + val_df['content'].fillna('')
test_df['text'] = test_df['headline'].fillna('') + ' ' + test_df['content'].fillna('')

# Label encoding
label2id = {'authentic': 0, 'fake': 1, 'ai_fake': 2}
id2label = {0: 'authentic', 1: 'fake', 2: 'ai_fake'}

train_df['label_id'] = train_df['label'].map(label2id)
val_df['label_id'] = val_df['label'].map(label2id)
test_df['label_id'] = test_df['label'].map(label2id)

# Remove NaN
train_df = train_df.dropna(subset=['text', 'label_id'])
val_df = val_df.dropna(subset=['text', 'label_id'])
test_df = test_df.dropna(subset=['text', 'label_id'])

print(" Data loaded!")
print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")

 Data loaded!
Train: 10500
Val:   2250
Test:  2250


In [6]:
# Load XLM-RoBERTa tokenizer
print("Loading XLM-RoBERTa tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
print(" Tokenizer loaded!")

# Dataset class
class BanglaNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
train_dataset = BanglaNewsDataset(
    train_df['text'].values,
    train_df['label_id'].values,
    tokenizer
)
val_dataset = BanglaNewsDataset(
    val_df['text'].values,
    val_df['label_id'].values,
    tokenizer
)
test_dataset = BanglaNewsDataset(
    test_df['text'].values,
    test_df['label_id'].values,
    tokenizer
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

print(" Datasets ready!")
print(f"Train batches: {len(train_loader)}")

Loading XLM-RoBERTa tokenizer...
 Tokenizer loaded!
 Datasets ready!
Train batches: 657


In [7]:
# Load XLM-RoBERTa model
print("Loading XLM-RoBERTa model...")
model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)
model = model.to(device)
print(" Model loaded!")

# Optimizer
from transformers import AdamW, get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_loader) * 3
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")

Loading XLM-RoBERTa model...


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Model loaded!
Total training steps: 1971


In [8]:
# Training function
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    
    for batch in loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    
    return total_loss / len(loader)

# Evaluation function
def evaluate(model, loader):
    model.eval()
    preds = []
    true_labels = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            pred = outputs.logits.argmax(dim=1)
            preds.extend(pred.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    return preds, true_labels

# Train for 3 epochs
print("Training started!")
best_f1 = 0

for epoch in range(3):
    print(f"\nEpoch {epoch+1}/3")
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler)
    print(f"Train Loss: {train_loss:.4f}")
    
    # Validate
    val_preds, val_true = evaluate(model, val_loader)
    val_f1 = f1_score(val_true, val_preds, average='macro')
    print(f"Val Macro F1: {val_f1*100:.2f}%")
    
    # Save best model
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(
            model.state_dict(),
            'C:/Users/Riyad/projects/fake_news/xlmroberta_best.pt'
        )
        print(f" Best model saved!")

print(f"\nBest Val F1: {best_f1*100:.2f}%")

Training started!

Epoch 1/3
Train Loss: 0.4905
Val Macro F1: 87.30%
 Best model saved!

Epoch 2/3
Train Loss: 0.3064
Val Macro F1: 88.88%
 Best model saved!

Epoch 3/3
Train Loss: 0.2357
Val Macro F1: 90.15%
 Best model saved!

Best Val F1: 90.15%


In [9]:
# Load best model
model.load_state_dict(torch.load(
    'C:/Users/Riyad/projects/fake_news/xlmroberta_best.pt'
))

# Test evaluation
test_preds, test_true = evaluate(model, test_loader)

print("=== XLM-RoBERTa Test Results ===")
print(classification_report(
    test_true,
    test_preds,
    target_names=['authentic', 'fake', 'ai_fake']
))

test_f1 = f1_score(test_true, test_preds, average='macro')
print(f"Test Macro F1: {test_f1*100:.2f}%")

=== XLM-RoBERTa Test Results ===
              precision    recall  f1-score   support

   authentic       0.82      0.91      0.86       750
        fake       0.88      0.80      0.84       750
     ai_fake       0.99      0.98      0.98       750

    accuracy                           0.90      2250
   macro avg       0.90      0.90      0.89      2250
weighted avg       0.90      0.90      0.89      2250

Test Macro F1: 89.50%
